In [37]:
!pip install kagglehub --quiet
import kagglehub

import cv2
import os
import numpy as np
from PIL import Image
from pathlib import Path
import pandas as pd

!pip install albumentations --quiet
import albumentations as A

DATASET_ROOT  = "quackphuc/egoblind-short-context-frames"
OUTPUT_PATH   = "/kaggle/working/egoblind_processed/"
SAMPLE_RATE   = 10
TARGET_SIZE   = (640, 640)
PADDING_COLOR = (114, 114, 114)

os.makedirs(f"{OUTPUT_PATH}/images", exist_ok=True)
os.makedirs(f"{OUTPUT_PATH}/images_augmented", exist_ok=True)

print(f"Output will be saved to: {OUTPUT_PATH}")

Output will be saved to: /kaggle/working/egoblind_processed/


In [38]:
# Load filtered CSV
df = pd.read_csv("/kaggle/input/datasets/atharvadhupkar/egoblind-filtered-csv/egoblind_for_review")

# Dataset root is the original EgoBlind dataset
csv_path = kagglehub.dataset_download(DATASET_ROOT)
DATASET_ROOT = csv_path

print(f"DATASET_ROOT: {DATASET_ROOT}")
print(f"Total rows: {len(df)}")
print(f"Unique folders: {df['frame_folder_path'].nunique()}")
print(f"\nSplit breakdown:")
print(df['frame_folder_path'].str.split('/').str[0].value_counts())
print(f"\nSample paths:")
print(df['frame_folder_path'].head())
print(df.head())

DATASET_ROOT: /kaggle/input/datasets/quackphuc/egoblind-short-context-frames
Total rows: 359
Unique folders: 359

Split breakdown:
frame_folder_path
train    359
Name: count, dtype: int64

Sample paths:
0     train/v_00000_2
1     train/v_00002_3
2     train/v_00003_3
3     train/v_00011_2
4    train/v_00014_12
Name: frame_folder_path, dtype: object
  question_id                                           question answer0  \
0   v_00000_2               Is everything secure inside the car?    Yes.   
1   v_00002_3                Is it safe to cross the street now?    Yes.   
2   v_00003_3                Is it safe to cross the street now?     No.   
3   v_00011_2   Is it safe for me to walk through the exit door?    Yes.   
4  v_00014_12  Are there any obstacles on the ground I need t...    Yes.   

                                             answer1  \
0                                                NaN   
1  Yes, but there may be vehicles coming and goin...   
2                  No, 

In [39]:
def letterbox_resize(frame, target_size=TARGET_SIZE, padding_color=PADDING_COLOR):
    h, w = frame.shape[:2]
    scale = min(target_size[0] / w, target_size[1] / h)
    new_w = int(w * scale)
    new_h = int(h * scale)

    resized = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    canvas = np.full((target_size[1], target_size[0], 3), padding_color, dtype=np.uint8)
    x_off = (target_size[0] - new_w) // 2
    y_off = (target_size[1] - new_h) // 2
    canvas[y_off:y_off + new_h, x_off:x_off + new_w] = resized

    return canvas


def preprocess_folder(folder_path, output_dir, sample_rate=SAMPLE_RATE):
    """
    Read all frames from a folder, letterbox resize, save to output_dir.
    Returns number of frames saved.
    """
    if not os.path.exists(folder_path):
        print(f"Folder not found {folder_path}")
        return 0

    frames = sorted([
        f for f in os.listdir(folder_path)
        if f.endswith((".jpg", ".png", ".jpeg"))
    ])

    if len(frames) == 0:
        print(f"No images found in {folder_path}")
        return 0

    folder_name = os.path.basename(folder_path)
    saved = 0

    for i, fname in enumerate(frames):
        if i % sample_rate != 0:
            continue

        frame     = np.array(Image.open(os.path.join(folder_path, fname)).convert("RGB"))
        processed = letterbox_resize(frame)

        out_name = f"{folder_name}_{fname}"
        cv2.imwrite(
            os.path.join(output_dir, out_name),
            cv2.cvtColor(processed, cv2.COLOR_RGB2BGR)
        )
        saved += 1

    return saved

print("Helper functions defined")

Helper functions defined


In [40]:
total_saved  = 0
folder_paths = df["frame_folder_path"].unique().tolist()

for i, rel_path in enumerate(folder_paths):
    full_path = os.path.join(DATASET_ROOT, rel_path)
    # print(f"[{i+1}/{len(folder_paths)}] {rel_path}")
    saved = preprocess_folder(full_path, f"{OUTPUT_PATH}images")
    total_saved += saved
    # print(f"  Saved {saved} frames\n")


print(f"DONE — Total frames saved: {total_saved}")
print(f"Stored in: {OUTPUT_PATH}/images")

DONE — Total frames saved: 359
Stored in: /kaggle/working/egoblind_processed//images


In [41]:
augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
    A.Rotate(limit=15, p=0.3),
])

frames     = os.listdir(f"{OUTPUT_PATH}/images")
aug_output = f"{OUTPUT_PATH}/images_augmented/"
aug_count  = 0

for fname in frames:
    if not fname.endswith(".jpg"):
        continue

    img       = np.array(Image.open(os.path.join(f"{OUTPUT_PATH}/images", fname)).convert("RGB"))
    augmented = augment(image=img)["image"]
    Image.fromarray(augmented).save(os.path.join(aug_output, fname))
    aug_count += 1

print(f"Augmented {aug_count} frames")
print(f"Stored in: {aug_output}")

Augmented 359 frames
Stored in: /kaggle/working/egoblind_processed//images_augmented/


In [42]:
images  = [f for f in os.listdir(f"{OUTPUT_PATH}/images") if f.endswith(".jpg")]
corrupt = []
sizes   = set()

for fname in images:
    try:
        img = Image.open(os.path.join(f"{OUTPUT_PATH}/images", fname))
        sizes.add(img.size)
    except:
        corrupt.append(fname)

print("=" * 40)
print(f"Total frames :  {len(images)}")
print(f"Corrupt files:  {len(corrupt)}")
print(f"Unique sizes :  {sizes}")  # should only be (640, 640)
print(f"Output path  :  {OUTPUT_PATH}")

if sizes == {(640, 640)}:
    print("\nAll frames are 640x640")
else:
    print("\nMixed sizes detected")

Total frames :  359
Corrupt files:  0
Unique sizes :  {(640, 640)}
Output path  :  /kaggle/working/egoblind_processed/

All frames are 640x640 — preprocessing complete!
